# Data Splitting for Semi-Automatic Label Generation

This notebook creates a representative sample for the semi-automatic label generation method. It prepares and analyzes CT colonography metadata, including filtering, merging, and demographic statistics. Stratified sampling ensures balanced representation by gender, age, and manufacturer, and the resulting distributions are visualized for quality control.

In [73]:
import pandas as pd

In [74]:
metadata_tcia = pd.read_excel("data/metadata_tcia.xlsx")
metadata_ku = pd.read_json("data/metadata_ku.json", lines=True)

print(f"TCIA data shape: {metadata_tcia.shape}")
print(f"KU data shape: {metadata_ku.shape}")

TCIA data shape: (3451, 36)
KU data shape: (1714, 25)


### Data exploration

In [75]:
# get Patient Sex count of data
group_by_patient = metadata_tcia.groupby('Patient ID').first().reset_index()
sex_counts = group_by_patient['Patient Sex'].value_counts()
sex_counts

Patient Sex
F    401
M    353
U     19
O     12
Name: count, dtype: int64

In [76]:
# count unknown values per column
unknown_counts = metadata_tcia.isnull().sum()
unknown_counts = unknown_counts[unknown_counts > 0]
print("Unknown values per column:")
print(unknown_counts)

Unknown values per column:
Patient Birth Date                         3451
Patient Sex                                 202
Ethnic Group                               3451
Study Description                           204
Admitting Diagnosis Description            3404
Study ID                                    194
Patient Age                                 429
Longitudinal Temporal Event Type           3451
Longitudinal Temporal Offset From Event    3451
Protocol Name                               261
Series Date                                 328
Series Description                          250
Annotations Flag                           3451
Manufacturer                                202
Manufacturer Model Name                     202
Software Versions                           203
Third Party Analysis                       3451
dtype: int64


In [77]:
# merge dataframes by 'Study Instance UID' and 'InstanceUID'
merged_data = pd.merge(metadata_tcia, metadata_ku, left_on='Series Instance UID', right_on='InstanceUID', how='inner')
# Display the first few rows of the merged data
print(merged_data.shape)
merged_data.head(2)


(1714, 61)


,Patient ID,Patient Name,Patient Birth Date,Patient Sex,Ethnic Group,Phantom,Species Code,Species Description,Study Instance UID,Study Date,...,patients_age,slice_location,gender,new_sub_id,scan,position,mha_path,dicom_path,split,segmentation_path
0,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,59.0,551.9,F,sub001,1,prone,converted/sub001/sub001_pos-prone_scan-1_conv-...,raw/sub001/sub001_pos-prone_scan-1.zip,None,None
1,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,59.0,562.2,F,sub001,1,supine,converted/sub001/sub001_pos-supine_scan-1_conv...,raw/sub001/sub001_pos-supine_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...


In [78]:
# only keep ffs, ffp, and hfs, hfp
def filter_position(position):
    if position in ['FFS', 'FFP', 'HFS', 'HFP']:
        return position
    else:
        return None
merged_data['patient_position'] = merged_data['patient_position'].apply(filter_position)
# Drop rows with None values in 'patient_position'
merged_data = merged_data[merged_data['patient_position'].notnull()]

print(merged_data.shape)

(1666, 61)


In [79]:
# count unknown values per column
unknown_counts = merged_data.isnull().sum()
unknown_counts = unknown_counts[unknown_counts > 0]
print("Unknown values per column:")
print(unknown_counts)

Unknown values per column:
Patient Birth Date                         1666
Patient Sex                                  83
Ethnic Group                               1666
Study Description                            85
Admitting Diagnosis Description            1642
Study ID                                     80
Patient Age                                 199
Longitudinal Temporal Event Type           1666
Longitudinal Temporal Offset From Event    1666
Protocol Name                               112
Series Date                                 140
Series Description                          106
Annotations Flag                           1666
Manufacturer                                 83
Manufacturer Model Name                      83
Software Versions                            84
Third Party Analysis                       1666
filename                                      5
volume                                        5
surface                                       5
time         

In [80]:
# min max age
merged_data = merged_data[merged_data['Patient Age'].notnull()]
min_age = merged_data['Patient Age'].min()
max_age = merged_data['Patient Age'].max()
print(f"Min age: {min_age}")
print(f"Max age: {max_age}")

Min age: 009Y
Max age: 110Y


In [81]:
# Extract numeric part from 'Patient Age' and convert to integers
merged_data['Patient Age'] = merged_data['Patient Age'].str.extract(r'(\d+)').astype(int)

# Keep only rows where 'Patient Age' is between 50 and 90
merged_data = merged_data[(merged_data['Patient Age'] >= 50) & (merged_data['Patient Age'] <= 90)]
# Print the shape of the filtered DataFrame
print(merged_data.shape)

(1401, 61)


In [82]:
merged_data['new_sub_id'] = merged_data['new_sub_id'].str.extract(r'sub(\d{3})')

In [83]:
merged_data['new_sub_id']

0       001
1       001
2       002
3       002
5       003
       ... 
1709    823
1710    824
1711    824
1712    825
1713    825
Name: new_sub_id, Length: 1401, dtype: object

In [84]:
# count unknown values per column
unknown_counts = merged_data.isnull().sum()
unknown_counts = unknown_counts[unknown_counts > 0]
print("Unknown values per column:")
print(unknown_counts)

Unknown values per column:
Patient Birth Date                         1401
Ethnic Group                               1401
Study Description                             2
Admitting Diagnosis Description            1401
Longitudinal Temporal Event Type           1401
Longitudinal Temporal Offset From Event    1401
Protocol Name                                29
Series Date                                  57
Series Description                           23
Annotations Flag                           1401
Software Versions                             1
Third Party Analysis                       1401
filename                                      3
volume                                        3
surface                                       3
time                                          3
collapsed                                     3
split                                      1041
segmentation_path                          1041
dtype: int64


In [85]:
# remove the ones with no Software Versions
merged_data = merged_data[merged_data['Software Versions'].notnull()]
# Print the shape of the filtered DataFrame
print(merged_data.shape)
merged_data.head(2)

(1400, 61)


,Patient ID,Patient Name,Patient Birth Date,Patient Sex,Ethnic Group,Phantom,Species Code,Species Description,Study Instance UID,Study Date,...,patients_age,slice_location,gender,new_sub_id,scan,position,mha_path,dicom_path,split,segmentation_path
0,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,59.0,551.9,F,001,1,prone,converted/sub001/sub001_pos-prone_scan-1_conv-...,raw/sub001/sub001_pos-prone_scan-1.zip,None,None
1,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,59.0,562.2,F,001,1,supine,converted/sub001/sub001_pos-supine_scan-1_conv...,raw/sub001/sub001_pos-supine_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...


In [86]:
merged_data.columns

Index(['Patient ID', 'Patient Name', 'Patient Birth Date', 'Patient Sex',
       'Ethnic Group', 'Phantom', 'Species Code', 'Species Description',
       'Study Instance UID', 'Study Date', 'Study Description',
       'Admitting Diagnosis Description', 'Study ID', 'Patient Age',
       'Longitudinal Temporal Event Type',
       'Longitudinal Temporal Offset From Event', 'Series Instance UID',
       'Project', 'Modality', 'Protocol Name', 'Series Date',
       'Series Description', 'Body Part Examined', 'Series Number',
       'Annotations Flag', 'Manufacturer', 'Manufacturer Model Name',
       'Software Versions', 'Image Count', 'Max Submission Timestamp',
       'License Name', 'License URI', 'Collection URI', 'File Size',
       'Date Released', 'Third Party Analysis', 'filename', 'volume',
       'surface', 'time', 'collapsed', 'name', 'InstanceUID', 'dim',
       'iop_list', 'patient_position', 'pixel_spacing', 'slice_thickness',
       'image_position_patient', 'patients_history

In [87]:
# Remove rows where 'Patient Sex' is 'O'
merged_data = merged_data[merged_data['Patient Sex'] != 'O']

# Print the shape of the filtered DataFrame
print(merged_data.shape)

(1398, 61)


In [88]:
# count rows per manufacturer
manufacturer_counts = merged_data['Manufacturer'].value_counts()
print(manufacturer_counts)

Manufacturer
SIEMENS               839
GE MEDICAL SYSTEMS    447
Philips                61
TOSHIBA                51
Name: count, dtype: int64


In [89]:
# get sex count per manufacturer grouped by patient id
merged_data_grouped_by_id = merged_data.groupby("Patient ID").first()
sex_counts_per_manufacturer = merged_data_grouped_by_id.groupby('Manufacturer')['Patient Sex'].value_counts()
sex_counts_per_manufacturer

Manufacturer        Patient Sex
GE MEDICAL SYSTEMS  F              116
                    M              104
Philips             F               19
                    M               11
SIEMENS             F              224
                    M              190
TOSHIBA             F               17
                    M                9
Name: count, dtype: int64

In [90]:
manufacturer_model_counts = merged_data_grouped_by_id['Manufacturer Model Name'].value_counts()
print(manufacturer_model_counts)

Manufacturer Model Name
Sensation 64         259
LightSpeed16         196
Sensation 16         155
Aquilion              26
Brilliance 40         25
LightSpeed VCT        21
Brilliance 64          5
LightSpeed Pro 16      3
Name: count, dtype: int64


In [91]:
# keep philips and toshiba as held out test sets
held_out_test = merged_data.loc[merged_data['Manufacturer'].isin(["Philips", "TOSHIBA"])]
print(held_out_test.shape)

# remove from merged_data
merged_data_train = merged_data.loc[~merged_data['Manufacturer'].isin(["Philips", "TOSHIBA"])]
print(merged_data.shape)
print(merged_data_train.shape)

(112, 61)
(1398, 61)
(1286, 61)


In [92]:
# make sure patients are separated
held_out_test_subject = set(held_out_test["Patient ID"])

subject_overlap = merged_data_train.loc[merged_data_train["Patient ID"].isin(held_out_test_subject)]
print(subject_overlap)


Empty DataFrame
Columns: [Patient ID, Patient Name, Patient Birth Date, Patient Sex, Ethnic Group, Phantom, Species Code, Species Description, Study Instance UID, Study Date, Study Description, Admitting Diagnosis Description, Study ID, Patient Age, Longitudinal Temporal Event Type, Longitudinal Temporal Offset From Event, Series Instance UID, Project, Modality, Protocol Name, Series Date, Series Description, Body Part Examined, Series Number, Annotations Flag, Manufacturer, Manufacturer Model Name, Software Versions, Image Count, Max Submission Timestamp, License Name, License URI, Collection URI, File Size, Date Released, Third Party Analysis, filename, volume, surface, time, collapsed, name, InstanceUID, dim, iop_list, patient_position, pixel_spacing, slice_thickness, image_position_patient, patients_history, sub_id, patients_age, slice_location, gender, new_sub_id, scan, position, mha_path, dicom_path, split, segmentation_path]
Index: []

[0 rows x 61 columns]


In [93]:
# create csv file and only keep filename

held_out_test[["name", "position", "new_sub_id"]].to_csv("filenames_held_out.csv", index=False)
merged_data_train[["name", "position", "new_sub_id"]].to_csv("filenames_train_val.csv", index=False)